# 🧩 Mini-Lab: First Win — Triage Your Own Text

**Module 1: Ship a Working AI Feature Today** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Run** a triage on your own text with no install, no account, and no API key
2. **Read** a structured JSON result with `id`, `label`, `summary`, and `latency_ms`
3. **Recognize** that a plausible mock label is not always a correct label

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Mock inference | A local stand-in for a model call: no API key, no network, and occasionally wrong |
| Structured JSON | A result with a stable shape: `id`, `label`, `summary`, `latency_ms` |


## Why This Matters

Mira is handed 4,000 unread support messages on a Friday. Leadership wants them sorted by morning: billing, bugs, account issues, feature requests, everything else.

You will not solve 4,000 rows today. You will do something better: **drive the whole path from "text goes in" to "label comes out" on a message you choose**, in the browser, with nothing to install.

**Privacy:** paste sample text, or personal-but-harmless text only. Never confidential or customer data — even into this local mock. The mock does not leave your session, but the habit starts now.

If this notebook will not open, download the `.ipynb` and run it in any Python 3 Jupyter environment. There is no account and no key.

Run the cells **top to bottom**. A later cell uses names defined above it.

In [1]:
import json
import time

from IPython.display import display, Markdown


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## A Stand-In for a Model Call

`classify_message` looks like an AI call: text in, structured result out. It is not a model. It is a **mock** — a keyword function that returns a plausible label so you can run the rest of the work with zero setup.

Allowed labels: `billing`, `bug`, `account`, `feature`, `other`.

In [2]:
LABELS = ("billing", "bug", "account", "feature", "other")

KEYWORD_RULES = (
    ("billing", ("invoice", "refund", "charged", "payment", "receipt", "subscription")),
    ("bug", ("crash", "error", "broken", "bug", "not working")),
    ("account", ("password", "login", "sign in", "locked out", "username")),
    ("feature", ("would be nice", "feature", "can you add", "wishlist")),
)


def mock_infer(text):
    """Return a plausible label and a short summary. Not always correct."""
    lowered = text.lower()
    label = "other"
    for name, keywords in KEYWORD_RULES:
        if any(keyword in lowered for keyword in keywords):
            label = name
            break
    summary = " ".join(text.split())
    if len(summary) > 100:
        summary = summary[:97] + "..."
    return {"label": label, "summary": summary}


def classify_message(text, msg_id="local-1"):
    """Take one message, call the mock, check the keys, return a JSON-ready dict."""
    started = time.perf_counter()
    cleaned = (text or "").strip()
    raw = mock_infer(cleaned)
    result = {
        "id": msg_id,
        "label": raw["label"],
        "summary": raw["summary"],
        "latency_ms": round((time.perf_counter() - started) * 1000, 2),
    }
    required = ("id", "label", "summary", "latency_ms")
    missing = [key for key in required if key not in result]
    if missing:
        raise ValueError(f"result is missing keys: {missing}")
    if result["label"] not in LABELS:
        raise ValueError(f"label {result['label']!r} is not in {LABELS}")
    return result


print("classify_message is ready. Allowed labels:", ", ".join(LABELS))

classify_message is ready. Allowed labels: billing, bug, account, feature, other


## Your First Win

The next cell is the whole move: **paste a message, run one cell, read JSON**.

Keep the sample, or replace `YOUR_TEXT` with your own non-confidential message. Then run the cell.

In [3]:
YOUR_TEXT = "I was charged twice for invoice 1842. Please refund the extra payment."

result = classify_message(YOUR_TEXT, msg_id="demo-001")
print(json.dumps(result, indent=2))

assert set(result) >= {"id", "label", "summary", "latency_ms"}
print("Shape check passed.")

{
  "id": "demo-001",
  "label": "billing",
  "summary": "I was charged twice for invoice 1842. Please refund the extra payment.",
  "latency_ms": 0.01
}
Shape check passed.


## Read the Four Fields

That printout is not a chat reply. It is a **record** — something you can store, count, and check.

In [4]:
md(
    f"| Field | Value | What it is |\n"
    f"|-------|-------|------------|\n"
    f"| `id` | `{result['id']}` | Which message this row is about |\n"
    f"| `label` | `{result['label']}` | The bucket the mock chose |\n"
    f"| `summary` | {result['summary']} | A short copy of the input |\n"
    f"| `latency_ms` | {result['latency_ms']} | How long the call took, in milliseconds |\n"
)

| Field | Value | What it is |
|-------|-------|------------|
| `id` | `demo-001` | Which message this row is about |
| `label` | `billing` | The bucket the mock chose |
| `summary` | I was charged twice for invoice 1842. Please refund the extra payment. | A short copy of the input |
| `latency_ms` | 0.01 | How long the call took, in milliseconds |


## The Mock Is Not Magic

Keyword rules pick the **first** matching bucket. That looks confident. It is still a guess.

The next cell is a real support-shaped message. Read it yourself before you run. What label would **you** give it?

In [5]:
tricky = "The refund button on the login page is broken."

tricky_result = classify_message(tricky, msg_id="demo-002")
print(json.dumps(tricky_result, indent=2))
print()
print("A human would likely call this a bug: a broken button.")
print("The mock saw 'refund' first and called it billing.")
print("A plausible label is not the same as a correct label.")

{
  "id": "demo-002",
  "label": "billing",
  "summary": "The refund button on the login page is broken.",
  "latency_ms": 0.01
}

A human would likely call this a bug: a broken button.
The mock saw 'refund' first and called it billing.
A plausible label is not the same as a correct label.


## 🎯 Summary

### Key Takeaways

1. **Text in, structured JSON out** — a usable AI feature returns a record (`id`, `label`, `summary`, `latency_ms`), not a chat paragraph
2. **A mock is enough to start** — you can run the path today with no install, no account, and no API key
3. **Plausible is not correct** — a confident JSON label can still be the wrong bucket; the mock is a stand-in, not a judge of quality